# Chapter 9 - Lab 3: <font color='blue'>Pricing a Care+ Extended Warranty with Parallel Data Enrichment</font> (customer-care edition)

**<font color='purple'>Goal</font>**:
Build a **pricing pipeline** that decides whether NovaMart should sell a Care+ extended
warranty to a customer, and at what annual price. The architecture is a **sequential + parallel
hybrid** (Chapter 7):

* **three enrichment agents** run in parallel with `asyncio.gather`: product reliability,
  the customer's claims history, and service-area risk;
* a **risk assessment** step then combines them into a risk score, premium, and an
  accept / refer decision.

The customer: **Rahul Verma** (C1002, Delhi) is buying a BreezeMax 1.5-ton inverter AC
(Rs 46,000) and asks for a 3-year Care+ plan.

| Finance (insurance) edition | Customer-care edition |
|---|---|
| Credit-bureau check | **Product reliability** (12-month failure rate of the model, average repair cost) |
| Driving record (violations, accidents) | **Customer claims history** (warranty claims, return rate, abuse flag) |
| Property valuation (value, risk zone) | **Service-area risk** (power fluctuation, technician coverage, climate) |
| Annual premium = 0.4% of property value x risk multiplier | Annual premium = 4% of device value x risk multiplier |

**<font color='purple'>Tech stack</font>**:

* **Python `asyncio`**: parallel enrichment via `asyncio.gather`.
* **No LLM calls** in the core path: the enrichment functions are deterministic stubs so the lab runs
  offline. The end of the notebook shows where LLM agents fit in.

## 1. Install packages

This lab uses only the standard library.

In [ ]:
# No additional packages needed for this lab.

## 2. Define the three enrichment functions

Each function simulates a slow internal API with `asyncio.sleep`. In production these would be calls to the service-analytics warehouse, the CRM and a field-service database; their *signatures* (input -> dict) stay the same, so you can swap them in without touching the orchestrator.

In [ ]:
import asyncio
import time


async def check_product_reliability(sku: str) -> dict:
    """Simulate the service-analytics lookup (parallel enrichment agent 1)."""
    await asyncio.sleep(1.0)  # Simulate API latency
    return {"sku": sku, "failure_rate_12m": 0.048,
            "avg_repair_cost_inr": 4200, "reliability_band": "good"}


async def check_claims_history(customer_id: str) -> dict:
    """Simulate the CRM warranty-claims lookup (parallel enrichment agent 2)."""
    await asyncio.sleep(1.2)
    return {"customer_id": customer_id, "warranty_claims_3yr": 1,
            "return_rate_12m": 0.0, "abuse_flag": False}


async def check_service_area(pincode: str) -> dict:
    """Simulate the service-area risk lookup (parallel enrichment agent 3)."""
    await asyncio.sleep(0.8)
    return {"pincode": pincode, "power_fluctuation": "high",
            "technician_coverage": "full", "climate": "extreme heat"}

## 3. Run all three enrichment calls in parallel

Wall-clock time is about `max(1.0, 1.2, 0.8) = 1.2 s` instead of `1.0 + 1.2 + 0.8 = 3.0 s`. At the checkout counter, that difference decides whether the cashier can quote a price while the customer waits.

In [ ]:
async def enrich_application(application: dict) -> dict:
    """Run three enrichment agents concurrently."""
    print("Running parallel data enrichment...")
    reliability, history, area = await asyncio.gather(
        check_product_reliability(application["sku"]),
        check_claims_history(application["customer_id"]),
        check_service_area(application["pincode"]),
    )
    print(f"  Product failure rate (12m): {reliability['failure_rate_12m']:.1%}")
    print(f"  Customer warranty claims (3y): {history['warranty_claims_3yr']}")
    print(f"  Service area: power fluctuation {area['power_fluctuation']}, "
          f"technicians {area['technician_coverage']}")
    return {"reliability": reliability, "history": history, "area": area}

## 4. Risk assessment: combining the enrichment data

The scoring is intentionally simple: a base score adjusted for product reliability, the customer's past claims and the local power quality, with a premium set as a share of the device value. A real insurer-retailer would use a calibrated actuarial model, but the *shape* of the pipeline (parallel enrichment -> synthesis) stays the same.

In [ ]:
def assess_risk(enrichment: dict, device_value: float) -> dict:
    """Synthesize enrichment data into a risk score, annual premium and decision."""
    failure_rate = enrichment["reliability"]["failure_rate_12m"]
    claims = enrichment["history"]["warranty_claims_3yr"]
    power = enrichment["area"]["power_fluctuation"]

    # Simplified risk scoring
    base_score = 100
    if failure_rate <= 0.05:
        base_score -= 10
    elif failure_rate > 0.08:
        base_score += 20
    base_score += claims * 15
    if power == "high":
        base_score += 10           # voltage swings damage compressors and boards
    if enrichment["history"]["abuse_flag"]:
        base_score += 40

    # Premium calculation (simplified)
    base_premium = device_value * 0.04  # 4% of the device value per year
    risk_multiplier = 1.0 + (base_score - 80) / 100

    return {
        "risk_score": base_score,
        "risk_tier": ("preferred" if base_score < 70
                      else "standard" if base_score < 100
                      else "high_risk"),
        "annual_premium_inr": round(base_premium * risk_multiplier, 2),
        "decision": ("accept" if base_score < 120
                     else "refer_to_senior"),
    }

## 5. Run the pricing pipeline

In [ ]:
application = {
    'application_id': 'CARE-APP-2026-014',
    'customer_id': 'C1002',
    'name': 'Rahul Verma',
    'product': 'BreezeMax 1.5 ton inverter AC',
    'sku': 'BMX-15',
    'device_value_inr': 46000,
    'pincode': '110017',
    'plan_requested': 'Care+ 3-year extended warranty',
}

print(f"Pricing Care+ for: {application['name']} - {application['product']}\n")

start = time.perf_counter()
enrichment = await enrich_application(application)
print(f'  (enrichment took {time.perf_counter() - start:.1f}s, not 3.0s)')

print('\nAssessing risk...')
risk = assess_risk(enrichment, application['device_value_inr'])

print('\n' + '=' * 40)
print('  CARE+ PRICING DECISION')
print('=' * 40)
print(f"  Risk score:     {risk['risk_score']}")
print(f"  Risk tier:      {risk['risk_tier']}")
print(f"  Annual premium: Rs {risk['annual_premium_inr']:,.2f}")
print(f"  Decision:       {risk['decision']}")

**Try it:** set `abuse_flag` to `True` in `check_claims_history` and re-run: the score jumps by
40 and the application is referred to a senior underwriter. Then set `power_fluctuation` to
`"normal"`. Which input moves the price most?

## 6. Where the LLMs would go in production

The version above is deliberately minimal. Two natural places for LLM agents:

1. **Document agent**: read the invoice, an installation photo, or a previous plan certificate
   into structured fields before enrichment (reuse the LlamaIndex pattern from Lab 1).
2. **Explanation agent**: write a two-sentence explanation the store executive can read to
   the customer ("your premium is higher because voltage fluctuation in your area...").
   Structured output (PydanticAI, Chapter 3) is a good fit.

Both slot into the same parallel-then-synthesise topology.

## 7. Results

**What to notice about this style:**

* **Parallelism is almost free** when the work is waiting on APIs, which pricing nearly always is.
* **Failure isolation is one flag away.** `asyncio.gather(..., return_exceptions=True)` lets you
  price with a missing input and decide what to do about it (for example, a default loading).
* **Sequential + parallel composes cleanly.** This lab's outer shape is the same as Lab 1's pipeline;
  parallel enrichment is just one step in a longer sequence.

**Discussion questions for class**

1. Is it fair to charge a customer more because of the power quality in their neighbourhood? What would a regulator ask?
2. The premium uses the customer's past claims. How could that discourage honest customers from claiming?
3. What would you A/B test before launching risk-based Care+ pricing?